# 응용 05 · 직접 학습한 last.pt의 val 평가

셀은 위에서 아래로 **Shift+Enter**로 실행합니다. 계산·표시에는 pandas가 필요합니다. 이 문서는 제공된 square·circle·triangle 도형 데이터만 다룹니다.

결과 파일 자체는 Git에 포함하지 않습니다. 대신 문서 작성 시 읽은 관측 수치를 코드 셀에 기록해, outputs가 없는 환경에서도 표와 계산을 볼 수 있게 했습니다. 저장된 수치는 과거 실행의 기록이며 문서를 다시 실행해 모델을 학습하는 것은 아닙니다. 실제 모델 호출은 Markdown 코드 예제로 제공합니다. 검증 여부·제출용 판단은 직접 작성하세요.

## 문서의 범위와 근거

이 폴더의 `05_validation.py`를 따라 체크포인트 읽기 → val 평가 → 전체 지표 → 클래스별 표 순서로 설명합니다. 사용 가중치는 `outputs/cli_smoke/weights/last.pt`, 관측 수치는 `outputs/validation_metrics.json`과 `outputs/class_metrics.csv`에 근거합니다.

## 1. 04번과 05번의 관계

04번은 정답을 이용해 모델 가중치를 갱신합니다. 05번은 저장된 `outputs/cli_smoke/weights/last.pt`를 다시 읽어 val 데이터에서 지표를 계산합니다. 04번에서 생성한 체크포인트를 사용합니다.

검증에는 이미지 12장, 객체 30개(클래스별 10개), 빈 라벨 이미지 2장이 있습니다. val은 설정·모델 비교에 사용하고 test는 최종 평가에 남겨둡니다. 05번에서는 test를 읽지 않습니다.

In [1]:
from pathlib import Path
import pandas as pd

roots = [Path.cwd(), *Path.cwd().parents]
BASE = next((root / "20261106/02_advanced" for root in roots
             if (root / "20261106/02_advanced/04_train.py").is_file()), None)
if BASE is None:
    raise FileNotFoundError("저장소 안에서 문서를 실행하세요.")
print("실습 폴더:", BASE)


실습 폴더: /home/swchoi/projects/python/practice_assignment/20261106/02_advanced


## 2. 구현을 단계별로 읽기

1. `yaml.safe_dump()`로 train·val 경로와 클래스 이름을 저장합니다.
2. `YOLO(str(out / "cli_smoke/weights/last.pt"))`로 체크포인트를 읽습니다.
3. `metrics = model.val(...)`로 평가하고 반환 객체를 변수에 저장합니다.
4. `metrics.box`에서 전체·클래스별 지표를 꺼냅니다.
5. JSON과 CSV로 저장하고 터미널에 요약을 표시합니다.

현재 구현은 평가 조건을 `imgsz=160`, `batch=4`, `workers=0`, `split="val"`로 명시합니다. 사용 가능한 GPU가 없으면 CPU를 선택합니다. 이번 문서 작성 중 재실행은 CPU에서 수행했습니다. 기존 학습 args의 GPU 기록과 구분합니다.

In [2]:
checkpoint = BASE / "outputs/cli_smoke/weights/last.pt"
print("모델 경로:", checkpoint)
print("평가 이미지:", BASE / "data/images/val")
print("저장 수치의 평가 장치: CPU")
print("평가 설정: split=val, imgsz=160, batch=4, workers=0")


모델 경로: /home/swchoi/projects/python/practice_assignment/20261106/02_advanced/outputs/cli_smoke/weights/last.pt
평가 이미지: /home/swchoi/projects/python/practice_assignment/20261106/02_advanced/data/images/val
저장 수치의 평가 장치: CPU
평가 설정: split=val, imgsz=160, batch=4, workers=0


### 실제 모델 호출 예시

아래는 파일 구현에서 사용하는 호출입니다. 실행하면 평가가 다시 수행되므로 설명용으로 제시합니다.

```python
metrics = model.val(
    data=str(path), split="val", imgsz=160, batch=4,
    device=0 if torch.cuda.is_available() else "cpu", workers=0,
    plots=False, project=str(out), name="validation", exist_ok=True
)
```

`metrics`는 단순한 숫자가 아니라 여러 지표가 들어 있는 결과 객체입니다. `.box.map50`과 `.box.map`을 읽은 뒤 `float()`로 Python 기본 숫자로 바꾸면 JSON에 저장할 수 있습니다.

## 3. precision·recall·AP·mAP 구분

|지표|의미|
|---|---|
|Precision|예측 중 정답과 연결된 비율 TP/(TP+FP)|
|Recall|전체 정답 중 찾은 비율 TP/(TP+FN)|
|AP50|IoU 0.50에서 클래스 하나의 Precision–Recall 관계 요약|
|AP50–95|IoU 0.50~0.95의 AP 평균|
|mAP50|클래스별 AP50 평균|
|mAP50–95|클래스·IoU에 걸친 AP 평균|

평가기의 precision·recall 출력은 내부 confidence 선택 정책에 따라 요약됩니다. 6번의 confidence 한 개를 고정한 수치와 자동으로 같은 조건이라고 보지 않습니다. 또한 이 라이브러리 평가를 앞서 작성한 교육용 AP101 함수와 완전히 동일한 구현이라고 가정하지 않습니다.

In [3]:
validation_summary = {'map50': 0.0, 'map50_95': 0.0, 'split': 'val'}
print(pd.Series(validation_summary).to_string())


map50       0.0
map50_95    0.0
split       val


## 4. 클래스별 결과와 support

관측값 출처는 `outputs/class_metrics.csv`입니다. support는 해당 클래스의 정답 객체 수이며, 예측 수와 다릅니다. 세 클래스 모두 객체 10개씩입니다.

`metrics.box.ap_class_index`는 평가된 클래스 번호 목록입니다. `enumerate()`의 i는 목록 안의 위치이고 cls는 실제 클래스 번호입니다. 따라서 `.class_result(i)`에 cls를 무조건 대신 넣지 않습니다.

In [4]:
class_records = [{'class_id': 0.0, 'name': 'square', 'precision': 0.0, 'recall': 0.0, 'ap50': 0.0, 'ap50_95': 0.0}, {'class_id': 1.0, 'name': 'circle', 'precision': 0.0, 'recall': 0.0, 'ap50': 0.0, 'ap50_95': 0.0}, {'class_id': 2.0, 'name': 'triangle', 'precision': 0.0, 'recall': 0.0, 'ap50': 0.0, 'ap50_95': 0.0}]
class_table = pd.DataFrame(class_records)
class_table["class_id"] = class_table["class_id"].astype(int)
class_table["support_objects"] = 10
print(class_table.to_string(index=False))


 class_id     name  precision  recall  ap50  ap50_95  support_objects
        0   square        0.0     0.0   0.0      0.0               10
        1   circle        0.0     0.0   0.0      0.0               10
        2 triangle        0.0     0.0   0.0      0.0               10


## 5. 반환 객체를 표로 바꾸는 Python 문법

```python
rows = []
for i, cls in enumerate(metrics.box.ap_class_index):
    pr, rc, ap50, ap = metrics.box.class_result(i)
    rows.append([int(cls), model.names[int(cls)],
                 float(pr), float(rc), float(ap50), float(ap)])
```

`pr, rc, ap50, ap = ...`는 네 값을 변수 네 개로 나누어 받습니다. `model.names[int(cls)]`는 클래스 번호에 해당하는 이름을 찾습니다. `rows.append()`로 한 행씩 모아 `pd.DataFrame(rows, columns=[...])`로 표를 만듭니다.

`to_csv(index=False)`는 표의 행 번호를 별도 열로 저장하지 않는다는 뜻입니다. `json.dumps(summary, indent=2)`는 요약 딕셔너리를 들여쓰기한 JSON 문자열로 바꿉니다.

## 6. 이번 관측 수치를 읽기

세 클래스의 precision·recall·AP50·AP50–95는 모두 0입니다. 전체 mAP50과 mAP50–95도 0입니다. 04번의 학습 중 val 지표 역시 두 epoch 모두 0으로 기록돼 있습니다.

이 값만으로는 예측 자체가 없었는지, 예측은 있었지만 위치·클래스 조건이 맞지 않았는지 구분할 수 없습니다. 06번에서 confidence별 검출 수·TP·FP·FN을 따로 기록해 이 부분을 더 살펴봅니다. 무작위 초기화와 짧은 학습이 영향을 줬을 가능성은 원인 가설이며 이번 단일 실행만으로 확정하지 않습니다.

## 7. 직접 기록

|항목|직접 작성|
|---|---|
|평가 실행·검증 여부와 근거||
|체크포인트 선택 이유||
|이번 수치로 설명할 수 있는 범위||
|다음 비교에서 고정할 조건||

참고한 로컬 파일: `05_validation.py`, `data/labels/val/`, `outputs/validation_metrics.json`, `outputs/class_metrics.csv`. 이 문서의 셀은 수치 기록을 재표시하며 모델 평가를 자동으로 다시 실행하지 않습니다.